# Eye-tracking datasets

Each loader initially uses the annotation units chosen by the source study:
words, multiword interest areas or characters. Some sources record observations
for individual participants; others provide averages across participants.
These source units are what the API calls the datum's **native annotation layer**.

Where fixation events and their spatial alignment are available, responses can
be recomputed on compatible units. See the unified [segmentation and
reaggregation tutorial](../tutorials/segmentation.ipynb).

## Prerequisites

```bash
pip install -e ".[models,frequency,readers,zuco,r]"
replica fetch provo
```

`models` supplies GPT-2 support; `frequency` supplies the word-frequency table
used for the unigram-surprisal control. Other loader dependencies vary by file
format. The mixed-model example requires R with `lme4`; see
[installation](../tutorials/getting_started.ipynb).

In [ ]:
import numpy as np

## 1. Explore the current dataset inventory

The table is generated from `available_datasets()` and `describe_dataset()`;
adding a registered loader automatically adds it here. This does not load or
download corpus files. Inspect a key with `describe_dataset(key)` or
`replica datasets --describe NAME`.

In [ ]:
import pandas as pd

from replica.dataset import available_datasets, describe_dataset, load_dataset

# Read the installed catalog without downloading or loading dataset files.
descriptions = [describe_dataset(name) for name in available_datasets()]
inventory = pd.DataFrame(
    [
        {"Dataset": d.name, "Loader": d.loader, "Measures": ", ".join(d.measures)}
        for d in descriptions
        if d.paradigm == "eye-tracking"
    ]
)
inventory

## 2. Inspect one dataset

Select a registry key from the inventory. `load_dataset()` reads from
`PSYCHO_DATA_DIR`, or the configured dataset directory; fetch the source first
with `replica fetch NAME`. The cells below use PROVO's word annotations and
duration values in milliseconds.

In [ ]:
DATASET_NAME = "provo"
ds = load_dataset(DATASET_NAME)
print(f"Dataset:  {ds.info.name}")
print(f"Language: {ds.info.language}")
print(f"Measures: {ds.measures}")
print(f"Default:  {ds.default_measure}")

### Stimuli and source annotation units

A `Stimulus` holds the text, an ID and annotation layers. The source study's
units are spans in that text. Its `"words"` layer can contain multiword interest
areas or adjacent words without spaces; it need not match whitespace
segmentation. Character-based releases use a `"characters"` layer.

In [ ]:
stimuli = ds.stimuli()
print(f"{len(stimuli)} stimuli\n")

# First stimulus
stim = stimuli[0]
print(f"ID:     {stim.stimulus_id}")
print(f"Words:  {len(stim.layers['words'])}")
print(f"Layers: {list(stim.layers.keys())}")
print(f"Text:   {stim.text[:120]}...")
print(f"\nFirst 10 words: {stim.unit_strings()[:10]}")

### Observations (`Datum` objects)

A `Datum` holds measurements for one stimulus and, where provided, one
participant. Its `values` align with `stimulus.layers[datum.layer]`.
`ds.data()` exposes these loader records; `ds.feature(name)` selects a stored
quantity for use as a target, predictor or control in an `EvaluationSpec`.
Feature resolution returns `FeatureValues` with values and observation identities.

Eye-tracking duration zeros are marked missing by default. Set `ds.drop_skips = False`
before reading or evaluating to retain observed zero-duration skips; see the
[skip-policy discussion](../tutorials/segmentation.ipynb#skip-policy).

In [ ]:
data = ds.data(measure=ds.default_measure)
print(f"{len(data)} Datum objects for measure '{ds.default_measure}'\n")

datum = data[0]
print(f"Stimulus:    {datum.stimulus.stimulus_id}")
print(f"Participant: {datum.participant_id}")
print(f"Measure:     {datum.measure_name}")
print(f"Shape:       {datum.values.shape}")

# Show values on the observation's annotation layer.
units = datum.stimulus.unit_strings(datum.layer)
print("\nSource-unit values (first 10):")
for w, v in list(zip(units, datum.values, strict=True))[:10]:
    val_str = f"{v:7.1f} ms" if not np.isnan(v) else "    NaN"
    print(f"  {w:15s} -> {val_str}")

### Reading time distribution

Quick look at the distribution of the default measure across all observations.

In [ ]:
all_values = np.concatenate([d.values for d in data])
observed = all_values[np.isfinite(all_values)]

print(f"Measure: {ds.default_measure}")
print(f"Total observations: {len(all_values):,}")
print(f"Finite observations: {len(observed):,}")
print(f"Mean:   {np.mean(observed):.1f} ms")
print(f"Median: {np.median(observed):.1f} ms")
print(f"Std:    {np.std(observed):.1f} ms")
print(f"Range:  [{np.min(observed):.0f}, {np.max(observed):.0f}] ms")

## 3. Evaluate a comprehension model

Test whether GPT-2's contextual surprisal predicts reading times beyond word
length and **unigram surprisal**. An `EvaluationSpec` separates predictors,
controls and targets. Both comparison models receive the controls; only the
full model receives contextual surprisal. Fixation durations are the responses;
the binary `first_pass` indicator is omitted.

Enable logging to see progress:

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s")

In [ ]:
from replica.analysis import AnalysisConfig, EvaluationSpec, evaluate
from replica.dataset.measures import CANONICAL_RT_MEASURES
from replica.feature import (
    HuggingFaceLM,
    ModelFeature,
    Surprisal,
    UnigramSurprisal,
    UnitLength,
    WordfreqSource,
)

# Fixation durations only; first_pass is a 0/1 flag.
durations = [m for m in ds.measures if m in CANONICAL_RT_MEASURES]

spec = EvaluationSpec(
    predictors=[ModelFeature(HuggingFaceLM("gpt2"), Surprisal())],
    controls=[UnitLength(), UnigramSurprisal(WordfreqSource(ds.info.language))],
    targets=[ds.feature(m) for m in durations],
)

results = evaluate(
    dataset=ds,
    spec=spec,
    analysis=AnalysisConfig(n_folds=5),  # Up to five folds per holdout axis.
)

In [ ]:
print(f"{ds.info.name} -- GPT-2 Surprisal (controls: word length, unigram surprisal)")
print("=" * 75)
for r in results:
    print(
        f"  {r.dv_name:20s}  delta_ll={r.delta_ll:+.2f}  "
        f"delta_ll/unit={r.delta_ll_per_unit:+.6f}  n={r.n_observations}"
    )

## 4. Dataset-specific options

Use `describe_dataset(NAME)` for constructor and loading options. A loaded
dataset exposes its annotation layers and `covariates(stimulus)`. For example,
PROVO and RastrOS provide human cloze norms; MECO has release, language and site
selectors; ZuCo provides eye-tracking and EEG views. Check the chosen source's
metadata and citation before configuring an analysis.